# Gerador de Dimensão: Representantes

**Objetivo:** criar a carga inicial da dimensão `dim_representantes`, contendo os 9 representantes fixos da estrutura comercial (3 Gerentes, 1 por país + 6 Supervisores, 1 por centro de distribuição).

**Dependências:**
- `src/utils/scd2_handler.py` (classe `SCD2Handler`, controle SCD Tipo 2)
- `src/utils/faker_helper.py` (classe `FakerHelper`, geração de nomes coerentes com o locale de cada país, incluindo simulação de pequenas inconsistências de qualidade de dados)

**Widget:** `forcar_recriacao` — permite re-executar este notebook durante testes/desenvolvimento.

**Saída:** arquivo JSON gravado na Landing Zone (`/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/representantes/`).

**Observação de modelagem:** a estrutura hierárquica (cargo, país, centro vinculado) é fixa e conhecida — apenas o **nome** de cada representante é gerado via Faker. Gerentes não possuem um centro de distribuição específico vinculado (`centro_vinculado = null`), pois supervisionam todos os centros do seu país, relação obtida via `pais` em vez de vínculo direto.

In [0]:
# Instalações e configurações iniciais

%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.scd2_handler import SCD2Handler
from utils.faker_helper import FakerHelper

print("Dependências carregadas com sucesso.")

In [0]:
# Widget

dbutils.widgets.dropdown("forcar_recriacao", "false", ["true", "false"])

forcar_recriacao = dbutils.widgets.get("forcar_recriacao") == "true"

print(f"Forçar recriação: {forcar_recriacao}")

In [0]:
# Estrutura de representantes 

representantes_estrutura = [
    # (representante_id, cargo, pais, centro_vinculado)
    ("REP001", "Gerente", "Brasil", None),
    ("REP002", "Supervisor", "Brasil", "São Paulo"),
    ("REP003", "Supervisor", "Brasil", "Rio de Janeiro"),
    ("REP004", "Supervisor", "Brasil", "Minas Gerais"),
    ("REP005", "Gerente", "Argentina", None),
    ("REP006", "Supervisor", "Argentina", "Buenos Aires"),
    ("REP007", "Gerente", "Mexico", None),
    ("REP008", "Supervisor", "Mexico", "Guadalajara"),
    ("REP009", "Supervisor", "Mexico", "Cidade do México"),
]

print(f"Total de representantes definidos: {len(representantes_estrutura)}")


In [0]:
# Criando nomes com faker

nomes_gerados = []

for rep_id, cargo, pais, centro in representantes_estrutura:
    fh = FakerHelper(pais=pais.lower())  # normaliza para minúsculo antes de usar
    nome = fh.gerar_nome()
    nomes_gerados.append((rep_id, nome, cargo, pais, centro))

colunas_representantes = ["representante_id", "nome", "cargo", "pais", "centro_vinculado"]

for linha in nomes_gerados:
    print(linha)

In [0]:
# Transformação em dataframe spark

df_representantes_cru = spark.createDataFrame(nomes_gerados, colunas_representantes)

df_representantes_cru.display()

In [0]:
# Aplicar o SCD2Handler

scd2 = SCD2Handler()
df_representantes_scd2 = scd2.iniciar_controle_scd2(df_representantes_cru)

df_representantes_scd2.display()

In [0]:
# Gravar o resultado como JSON na Landing Zone

caminho_landing = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/representantes"

arquivos_existentes = []
try:
    arquivos_existentes = dbutils.fs.ls(caminho_landing)
except Exception:
    pass

if len(arquivos_existentes) > 0 and not forcar_recriacao:
    print("Já existem arquivos na Landing Zone de representantes. "
          "Nenhuma ação foi tomada (use o widget 'forcar_recriacao' = true para sobrescrever).")
else:
    (
        df_representantes_scd2
        .coalesce(1)
        .write
        .mode("overwrite")
        .json(caminho_landing)
    )
    print(f"Arquivo gravado com sucesso em: {caminho_landing}")